# Lecture 12 — More Nested Loops
### EES 3350/5350 · Python in Earth Science · Wednesday, September 30, 2026

On Monday  you met nested loops with the Calwood gullies and the soil scenarios. Today we pracitce them with planet data and add a couple new moves.

**Learning objectives for today**
- predict how many times each line of a nested loop runs
- use the nested-looop recipe (fresh list inside the outer loop) on a new dataset
- see how a loop turns five copy-pased plotting lines into three, and how *where* the plot call sits changes the picture
- write an inner loop whose range **depends on the outer loop**, to visit every unique pair exactly once
- print a table with a nested loop

---

## Part 0 : setup

Let's rebuild the `planets` dictinoary and `blackbody_demp()` from Lectures 05-07 with two new keys: each planet's average distance from the Sun (`distance_mkm`, in millions of km) and the blackbody temperature NASA lists (`nasa_bb_temp_K`), so we can check our function against theirs.

In [ ]:
planets = {
    "Mercury": {"solar_irradiance": 9082.7, "albedo": 0.068, "radius_km": 2439.7,
                "distance_mkm": 57.909,  "nasa_bb_temp_K": 439.6},
    "Venus":   {"solar_irradiance": 2601.3, "albedo": 0.77,  "radius_km": 6051.8,
                "distance_mkm": 108.210, "nasa_bb_temp_K": 226.6},
    "Earth":   {"solar_irradiance": 1361.0, "albedo": 0.294, "radius_km": 6371.0,
                "distance_mkm": 149.598, "nasa_bb_temp_K": 254.0},
    "Mars":    {"solar_irradiance": 586.2,  "albedo": 0.250, "radius_km": 3389.5,
                "distance_mkm": 227.956, "nasa_bb_temp_K": 209.8},
    "Jupiter": {"solar_irradiance": 50.26,  "albedo": 0.343, "radius_km": 69911,
                "distance_mkm": 778.479, "nasa_bb_temp_K": 109.9},
}

def blackbody_temp(S, a, stef_boltz=5.67e-8):
    """Predict a planet's blackbody surface temperature (K) from its solar constant (S) and albedo (a)."""
    T = ((1 - a) * S / (4 * stef_boltz)) ** (1/4)
    return T

First, a single loop to make sure our function agrees with NASA:

<div class="alert alert-success">

**Tip: one f-string, many placeholders.** An f-string can hold as many `{ }` placeholders as you want, each with its own formatting. Everything *outside* the braces (like `: ours = `, ` K | `) prints exactly as typed. Here's what each placeholder in that line does:

| **placeholder** | **means** | **prints** |
|:-|:-|:-|
| `{planet:>7}` | right-align the text in 7 spaces | `  Venus` |
| `{T:6.1f}` | 1 decimal place, 6 spaces wide | ` 226.6` |
| `{props['nasa_bb_temp_K']:6.1f}` | same, but for a dictionary lookup | ` 226.6` |
| `{diff:+5.1f}` | 1 decimal place, 5 spaces wide, **always show the sign** | ` +0.0` |

Setting widths is what makes the columns line up from one loop pass to the next.

**Watch your quotes:** the f-string uses double quotes (`f"..."`), so use **single** quotes for the dictionary key inside the braces: `props['nasa_bb_temp_K']`. In Python 3.12 and newer, matching double quotes also work, but in older versions they end the string early and cause a `SyntaxError`. Mixing the two kinds of quotes works everywhere.

</div>

In [ ]:
for planet, props in planets.items():
    T = blackbody_temp(props["solar_irradiance"], props["albedo"])
    diff = T - props["nasa_bb_temp_K"]
    print(f"{planet:>7}: ours = {T:6.1f} K | NASA = {props["nasa_bb_temp_K"]:6.1f} K | difference = {diff:+5.1f} K")

<div class="alert alert-warning">

**Question:** Four planets agree with NASA to within 0.1 K. Which one doesn't? Is our function wrong?
    
</div>

> your answer here

---
## Part 1 : the recipe, on new data

How would each planet's temperature change if its albedo were different? Let's sweep albedo from 0.0 (absorbs all sunlight) to 0.9 (reflects 90% of it) for **every** planet. First, the albedo values, counted with whole numbers and divided (Lectures 10 & 11a):

In [ ]:
albedos = []

for i in range(10):
    albedos.append(i / 10)

print(albedos)

Now Monday's recipe, step by step. Compare it to `fs_by_scenario` from Lecture 11: only the names and the science have changed.

In [ ]:
temps_by_planet = {} # first, make an empty results dictionary

for planet, props in planets.items(): # outer loop: one pass per planet
    temps = []                        # a fresh emtpy list for this planet
    for albedo in albedos:            # inner loop: one pass per albedo
        T = blackbody_temp(props["solar_irradiance"], albedo)
        temps.append(T)
    temps_by_planet[planet] = temps   # after the inner loop: store the list

# and some print statements
for planet, temps in temps_by_planet.items():
    print(f"{planet:>7}: {len(temps)} temperatures, from {max(temps):.0f} K down to {min(temps):.0f} K")

<div class="alert alert-success">

**Tip**: notice that we passed `albedo` (the loop variable) to `blackbody_temp()`, **not** `props["albedo"]` (the planet's real albedo). That's the whole point of a sweep: we hold the planet's sunlight fixed and ask "what if the albedo were different?"

</div>

<div class="alert alert-info">

### Exercise 1 : could liquid water survive?

Pure water freezes at **273.15 K**. For each planet, count how many of the 10 albedo values give a blackbody temperatrue **above** freezing. Print one loine per planet, like `Mercury: 9 of 10 albedos above freezing`.

*Where does the counter have to be created?*
    
</div>

In [ ]:
# your answer here
freezing_K = 273.15

for planet, temps in temps_by_planet.items():
    n_above = _______
    for T in _____:
        if T ___ freezing_K:
            n_above ______
    print(f"{planet:>7}: {n_above} of {len(temps)} albedos above freezing")

---
## Part 2 : a first look at plotting with loops

This is a preview. You saw `matplotlib` briefly in Lecture 06, and we'll learn plotting properly later. For today, don't worry about memorizing plotting commands. **Watch where the plot call sits in the loop**. That's the take away here.

Here's the result of our sweep for all five planets, the copy-and-paste way:

In [ ]:
import matplotlib.pyplot as plt

plt.plot(albedos, temps_by_planet["Mercury"], '.-', label="Mercury")
plt.plot(albedos, temps_by_planet["Venus"], '.-', label="Venus")
plt.plot(albedos, temps_by_planet["Earth"], '.-', label="Earth")
plt.plot(albedos, temps_by_planet["Mars"], '.-', label="Mars")
plt.plot(albedos, temps_by_planet["Jupiter"], '.-', label="Jupiter")

plt.axhline(273.15, color="gray", linestyle="--", label="water freezes")   # a horizontal line across the plot
plt.xlabel("albedo")
plt.ylabel("blackbody temperature (K)")
plt.legend()
plt.show()

It works! But it's the same feeling as in Lecture 09: five nearly identical lines, where only the planet changes. Adding Saturn would mean a sixth copy. Good thing we now know about loops! **"Do the same thing for each item in a collection" is what they do.**

In [ ]:
for planet, temps in temps_by_planet.items():
    plt.plot(albedos, temps, '.-', label=planet)

plt.axhline(273.15, color="gray", linestyle="--", label="water freezes")   # a horizontal line across the plot
plt.xlabel("albedo")
plt.ylabel("blackbody temperature (K)")
plt.legend()
plt.show()

Three lines of one loop replace five lines of copy-paste that you have to then modify. And the loop could draw 50 planets just as easily as 5. Notice what's *outside*, after every line is drawn.

You can also read the answer to Exercise 1 right off the plot: count how many points of each curve sit above the dashed line.

<div class="alert alert-info">

## Exercise 2 : predict the picture

Below, `plt.plot()` has moved into the **inner** loop, and it now plots one `(albedo, T)` point at a time. Before you run it, predict:

(a) how many times `plt.plot()` gets called, and

(b) what the picture will look like. Will you be able to tell which dots belong to which planet?
    
</div>

In [ ]:
for planet, temps in temps_by_planet.items():
    for albedo, T in zip(albedos, temps):
        plt.plot(albedo, T, "o")  # plotting inside an inner loop now

plt.xlabel("albedo")
plt.ylabel("blackbody temperatrue (K)")
plt.show()

<div class="alert alert-info">

## Exercise 3 : just the rocky planets

Jupiter is a gas giant, so it doesn't really have a surface. Plopt the albedo sweep for only the four **rocky** planets, using the list `rocky` below. Keep the dashed freezing line.

*Hint: you're looping over a list of names now, not `.items()`, so look up each planet's temperatures in `temps_by_planet`.
    
</div>

In [ ]:
# your answer here
rocky = ["Mercury", "Venus", "Earth", "Mars"]

for planet in _____:
    plt.plot(albedos, temps_by_planet[_____], label=______)

plt.axhline(273.15, color="gray", linestyle="--", label="water freezes")
plt.xlabel("albedo")
plt.ylabel("black-body temperature (K)")
plt.legend()
plt.show()

---
## Part 3 : when the inner loop depends on the outer loop

Sunlight spreads out as it travels, so solar irradiance should fall off with the **square** of distance from the Sun (the inverse-square law). IF that's true, then for any two planets A and B:
$$ \frac{S_A}{S_B} = \left(\frac{d_B}{d_A}\right)^2$$

Let's check it with NASA's numbers for **every pair** of planets. First attempt: a loop over planets inside a loop over planets.

In [ ]:
names = list(planets.keys())

n_lines = 0
for name_a in names:
    for name_b in names:
        print(name_a, "vs", name_b)
        n_lines += 1

print(n_lines, "comparisons")

That printed 25 comparisons, but two kinds of wasted work:
- **Every planet gets compared to itself** (`Mercury vs Mercury`). The ratio is always 1.
- **Every pair shows up twice** (`Mercury vs Venus` *and* `Venus vs Mercury`). Same comparison, flipped.

There are only **10** truly different pairs. To get exactly those, we loop over **positions** isntead, and start the inner loop one step *after* wherever the outerloop is:

In [ ]:
for i in range(len(names)):
    for j in range(i + 1, len(names)): # the inner range STARTS where the outer loop is plus 1
        print(f"i = {i}, j = {j}: {names[i]} vs {names[j]}")

#### What just happened
`range(i+1, len(names))` means "from `i + 1` up to, but not including 5". Because it uses `i`, **the inner loop's range is different on every outer pass:**

| **outer pass** | `i` | `names[i]` | `range(i + 1, 5)` | `j` **values** | **pairs** | **how many** |
|--:|--|--|--|--|--|--:|
|1 |0 |Mercury |`range(1, 5)` | 1, 2, 3, 4| Mercury with Venus, Earth, Mars, and Jupiter|4 |
|2 |1 |Venus |`range(2, 5)` | 2, 3, 4| Venus with Earth, Mars, and Jupiter|3 |
|3 |2 |Earth |`range(3, 5)` | 3, 4| Earth with Mars, and Jupiter|2 |
|4 |3 |Mars |`range(4, 5)` | 4| Mars with Jupiter|1 |
|5 |4 |Jupiter |`range(5, 5)` | (*none*)| (*the inner loop runs zero times*)|0 |

Each planet only pairs with planets **after** it in the list, so nothing is compared to itself and no pair is repeated. The total is `4 + 3 + 2 + 1 + 0 = 10`. In general, `n` items have `n x (n-1) / 2` unique pairs: `5 x 4 / 2 = 10`

<div class="alert alert-success">

**Tip**: on the last outer pass, `range(5, 5)` is **empty**, and a `for` loop over an empty range simply runs zero times. No error, nothing printed.

</div>

Now the science. For each unique pair, compare the measured irradiance ratio to the one the inverse-square law predicts:

In [ ]:
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a = planets[names[i]]
        b = planets[names[j]]
        measured = a["solar_irradiance"] / b["solar_irradiance"]
        predicted = (b["distance_mkm"] / a["distance_mkm"])**2
        print(f"{names[i]:>7} vs {names[j]:<7} | measured S ratio = {measured:7.2f} | (d_B/d_A)^2 = {predicted:7.2f}")

All 10 pairs agree to within rounding. The inverse-square law holds all the way out to Jupiter!

#### **When would something like this be useful?**
- **distances between sites**
    - Which field sites, sample locations, or earthquake epicenters are within 10 km of each other?
    - Think back to Lab 2, where we had four gullies that give 6 unique pairs. You did this by hand one at a time. Now you can do it using nested loops.
- **finding duplicates in a catalog**
    - earthquake catalogs that merge several agencies often list the same event twice. The check is to flag any pair of events within a few seconds and a few kilometers of each other.
    - landslide inventories built from multiple sources have the same problem.
    - comparing an event with itself would flag everything, which makes the `i + 1` start matter
- **checking stratigraphic consistency**
    - for every pair of dated layers in a section, the lower layer should be older, right? one loop over unique pairs finds any age inversions which ocould mean reworked material, a contaminated sample, or a fault,
- **rates between dated surfaces**
    - given terrace heights and ages, the incision rate between every pair of terraces shows whether incision was steady or episodic. 
- **comparing stations**
    - how simiolar is monthly rainfall at every pair of weather stations, or discharge at every pair of stream gagues?
- **satellite image pairs**
    - processing workflow loops over pairs of dates and keeping the ones within a set timegap helps us measure ground deformation over landslides and volcanoes. 

<div class="alert alert-info">

## Exercise 4 : Debug me! (no error message)

This loop was supposed to print the 10 unique pairs, but it prints **15** lines. Run it, figure out which extra lines snuck in, fix it by changing **one** thing.
    
</div>

In [ ]:
# this cell runs, but something is wrong. Find and fix the bug.
n_pairs = 0
for i in range(len(names)):
    for j in range(i, len(names)):
        print(names[i], "vs", names[j])
        n_pairs += 1

print (n_pairs, "pairs")

<div class="alert alert-info">

## Exercise 5 : planets with similar temperatures

**Help level: comments only**

Find every unique pair of planets whose blackbody temperatures (at their **real** albedos) are within **50 K** of each other. Print each such pair and its temperature difference.

*Hint: `abs()` gives the size difference without its sign: `abs(3-7)` is `4`.
    
</div>

In [ ]:
# your answer here
# loop over positions i
    # loop over positions j, starting one after i
        # compute the blackbody temperature of planet i and planet j (use each one's real albedo)
        # if the absolute difference is 50 K or less, print the pair and the difference

---
## Part 5 : printing a table

On Monday you drew a grid of stars with `print("*", end=" ")`. The same trick turns `temps_by_planet` into a table: **outer loop = rows (planets, inner loop = columns (albedos)**. Here's the reader row done for you:

In [ ]:
print(f"{'albedo:':>8}", end=" ")

for albedo in albedos:
    print(f"{albedo::>3}", end=" ")
print()

<div class="alert alert-info">

### Exercise 6 : an above freezing table

**Help level: on your own**.

Copy the header code, then add a nested loop that prints one row per planet: the planet's name, then for each albedo a `+` if that temperature is above 273.15, or a `.` if it isn't. The finished table should look like this:

```
 albedo: 0.0 0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 
Mercury:   +   +   +   +   +   +   +   +   +   . 
```

*Think about hwere the name, the symbols, and the bare `print()` each have to go.*
</div>

In [ ]:
# your answer here

<div class="alert alert-info">
    
### Exercise 7 : plot the inverse-square check

Instead of *reading* 10 printed ratios, **plot** them. Use the uniqe-pairs loop to build two lists: `measured_ratios` and `predicted_ratios`, then plot measured (y) against predicted (x) as dots. Add a dashed 1:1 line; if the law holds, every dot should sit on it.

The ratios run from about 2 to about 180, so use `plt.loglog()` instead of plt.plot() to give both axes a logarithmic scale.

</div>

In [ ]:
# your answer here
measured_ratios = []
predicted_ratios = []

# fill both lists with a unique-pairs nested loop


plt.loglog(predicted_ratios, measured_ratios, "o", label="planet pairs")
plt.loglog([1, 200], [1, 200], "k--", label="1:1 line")
plt.xlabel("predicted ratio, (d_B / d_A)^2")
plt.ylabel("measured ratio, S_A / S_B")
plt.legend()
plt.show()

## A useful table
| **idea** | **remember** |
|:-|:-|
| the recipe | fresh list/counter **inside the outer loop, before the inner loop**; store it **after** the inner loop |
| plotting with a loop | one `plt.plot()` per curve goes in the loop that walks the **curves**; labels, legend, and `plt.show()` go **after** the loop |
| too many plot calls | a plot call in the inner loop draws one point per pass, and every call gets a new color |
| unique pairs | `for i in range(n):` then `for j in range(i + 1, n):` |
| how many unique pairs | `n × (n − 1) / 2` |
| an empty range | `range(5, 5)` is empty, so the loop body runs zero times (no error) |
| tables | row label before the inner loop, cells inside it, bare `print()` after it |